# 2. Serve, evaluate, benchmark
Run in a **fresh runtime** (A100) after notebook 1 finished. Compares fp16 vs AWQ (vLLM) vs GGUF (llama.cpp).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Banking_FInetuning
!pip install -q vllm -r requirements-eval.txt

In [ ]:
import subprocess, time, requests, os, signal

def start(cmd):
    p = subprocess.Popen(cmd, stdout=open('/content/server.log','w'), stderr=subprocess.STDOUT, preexec_fn=os.setsid)
    for _ in range(180):
        try:
            if requests.get('http://localhost:8000/v1/models', timeout=2).ok:
                return p
        except Exception:
            pass
        time.sleep(2)
    raise RuntimeError(open('/content/server.log').read()[-3000:])

def stop(p):
    os.killpg(os.getpgid(p.pid), signal.SIGTERM); time.sleep(8)

def vllm_cmd(path, quant=None):
    c = ['python','-m','vllm.entrypoints.openai.api_server','--model',path,
         '--served-model-name','banking-triage','--max-model-len','1024','--port','8000']
    return c + (['--quantization',quant] if quant else [])

## Variant 1: fp16 baseline

In [ ]:
p = start(vllm_cmd('outputs/merged-fp16'))
!python -m src.eval.evaluate --model banking-triage --out results/eval-fp16.json --mlflow-run fp16 --concurrency 32
!python loadtest/bench.py --model banking-triage --label fp16
stop(p)

## Variant 2: AWQ int4

In [ ]:
p = start(vllm_cmd('outputs/merged-awq', 'awq'))
!python -m src.eval.evaluate --model banking-triage --out results/eval-awq.json --mlflow-run awq-int4 --concurrency 32
!python loadtest/bench.py --model banking-triage --label awq-int4
stop(p)

## Variant 3: GGUF Q4_K_M via llama.cpp server (GPU)
Builds llama-server with CUDA for the A100 (~8 min).

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=80 -DLLAMA_CURL=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-server -j8 > /dev/null

In [ ]:
p = start(['/content/llama.cpp/build/bin/llama-server','-m','outputs/gguf/model-Q4_K_M.gguf',
           '--alias','banking-triage','--port','8000','-ngl','99','-np','32','-c','32768'])
!python -m src.eval.evaluate --model banking-triage --out results/eval-gguf.json --mlflow-run gguf-q4km --concurrency 32
!python loadtest/bench.py --model banking-triage --label gguf-q4km
stop(p)

## Results table
Set `HOURLY` to your real GPU cost. Colab: compute units/hour for A100 (see Colab's resource panel) x your $/unit.

In [ ]:
import json, pandas as pd
rows = []
for name in ['fp16','awq','gguf']:
    m = json.load(open(f'results/eval-{name}.json')); m['variant'] = name; rows.append(m)
display(pd.DataFrame(rows).set_index('variant').round(4))
display(pd.read_csv('results/bench.csv').round(3))

In [ ]:
HOURLY = 1.2   # <- edit: A100 $/hour on your Colab plan
!python scripts/cost.py --hourly {HOURLY}